# Part 2, Topic 1: Introduction to Voltage Glitching (MAIN)

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *Similarly to clock glitching, inserting brief glitches into the power line of an embedded device can result in skipped instructions and corrupted results. Besides providing a more reliable glitch on some targets when compared to clock glitching, voltage glitching also has the advanatage that the Vcc pins on chips are more commonly accessable. This won't be covered in this course, but it can also be used to glitch a device asynchronous to its clock.*

**LEARNING OUTCOMES:**

* Understanding voltage glitch settings
* Building a voltage glitch and crash map.
* Modifying glitch circuit to increase glitch success

---
## Hands-on edition: how to use this notebook

This notebook is the original lab **plus seven short experiments** where *you* change the numbers and watch what a voltage glitch does to a real chip. Run a cell, change a number, run it again. A glitch that crashes the target is a normal outcome: the notebook reboots the target and carries on.

| You see | It means |
|---|---|
| a code cell that starts with **`>>> EDIT ME <<<`** | Change the numbers between the two `>>>` lines, then run the cell (**Shift+Enter**). Run it again with new numbers as often as you like. |
| a code cell that starts with **`RUN ONLY (do not edit)`** | Helper code. Run it once (top to bottom) and leave it alone. |
| a short markdown cell titled **My prediction** | Write down what you *expect* **before** you run the cell, and what you *saw* afterwards. Double-click the cell to edit it. |

**Before you start (hardware checklist)**

1. ChipWhisperer-**Husky** plugged straight into a USB port of the computer (not through a hub).
2. **CW312T-SAM4S** target seated on the **CW313** board, and the CW313 connected to the Husky with the ribbon cable.
3. **The Husky "Crowbar" SMA cable** connected to the **second SMA socket** of the CW313. This is the cable that actually shorts the target's power rail. *Without it every glitch does nothing: you will see only `normal` and no `success` at all.*
4. No other notebook or program using the Husky (**File > Shut Down Notebook** in any old notebook).

**The numbers in the EDIT ME cells are for the Husky.** The Husky measures glitch *width* (0 to 2296) and *offset* (0 to 4592) in small steps of its clock. A CW-Lite or CW308 works in percent of a clock cycle instead (see the full-search cell later for the values used for those boards).

Each experiment tells you what to edit and what you should see. The **"If something goes wrong"** section at the end of the notebook lists the common problems and their fixes. The numbers quoted as "what you should see" come from one particular kit: yours may differ by a few steps, so write down what *you* get.

## Voltage Glitch Hardware

The ChipWhisperer uses the same hardware block for both voltage and clock glitching, with the only difference being where the glitch output is routed to. Instead of routing to HS2, voltage glitching is performed by routing the glitch to either the `glitch_hp` transistor or the `glitch_lp` transistor. This can be done via the following API calls:

```python
scope.io.glitch_hp = True #enable HP glitch
scope.io.glitch_hp = False #disable LP glitch
scope.io.glitch_lp = True #enable LP glitch
scope.io.glitch_lp = False #disable LP glitch
```

While the hardware block are the same, you'll need to change how it's configued. You wouldn't want to try routing `"clock_xor"` to the glitch transistor and oscillate Vcc like the device's clock! Instead, the following two output settings are best suited to voltage glitching:

1. `"glitch_only"` - insert a glitch for a portion of a clock cycle based on `scope.glitch.width` and `scope.glitch.offset`
1. `"enable_only"` - insert a glitch for an entire clock cycle

Typically, the `"enable_only"` setting will be too powerful for most devices. One situation where it outshines `"glitch_only"` is in glitching asychronous to the target's clock. An example of this is glitching a target with an internal clock. In this case, the ChipWhisperer's clock can be boosted far above the target's to insert a precise glitch, with `repeat` functioning as `width` and `ext_offset` functioning as `offset`.

### Voltage Glitching vs. Clock Glitching

Voltage glitching has some obvious benefits over clock glitching, such as working for a wider varitey of targets, but its downsides are less obvious. One of the biggest is how much it depends on the actual glitch circuit itself. With clock glitching, it's relatively easy to insert a glitch - there's nothing external trying to keep the clock at a certain voltage level. This is very different for a target's power pins. When we try to drop the power pin to ground, there's a lot of stuff fighting us to keep the power pin at the correct voltage, such as decoupling capacitors, bulk supply capacitors, and the voltage regulator supplying the voltage. This means when we make small changes to the glitch circuit, the glitch settings and even our ability to insert a glitch at all completely change! Consider glitching a target on the CW308 UFO board. If you switch your coaxial cable length from 20cm to 40 cm, you'll need to find entirely new glitch settings to repeat the attack (if it's still even possible). This is quite easy to see on an oscilloscope or  using the ChipWhisperer's ADC: longer cables and lower valued shunt resistors will make the glitch less sharp and increase ringing.

While your first thought might be to go for as sharp a glitch as possible, this often won't result in a high glitch success rate. If you're unable to find any working glitches with your current setup, it might be worth changing you hardware setup a bit. For example, on the ChipWhisperer Lite 1 part, you can desolder SJ5 and solder header pins to JP6. Even just connecting these pins with a jumper will have different glitch behaviour than with a soldered SJ5.

You can refer to the training slides for more information about finding good glitch settings, as well as more on the theory side of voltage glitching.

### The Lab

To introduce you to volatge glitching and find some settings, we're going to walk back through the clock glitching loop lab. You may want to capture some power traces while you're first experimenting with glitches to see what effects different glitch widths have on the power trace. Another thing to keep in mind is that targets often won't tolerate the Vcc pin dropping for an extended period of time without crashing - once you see the target start to crash, you won't see much else with larger widths.

One thing you might have to change is the glitch repeat value. Depending on how wide your glitch is, the voltage at the power pin may not recover by the time the next glitch is inserted. This can have to effect of increasing subsequent glitches' strength, which may or may not be desirable. Since glitches inserted with repeat > 1 have different strength, it's a good idea to scan through ext_offset as well.

###  Higher Frequency Glitching

The XMEGA target, and to a lesser extent the STM32F3, is very difficult to glitch with the default ChipWhisperer settings. Try bumping the clock frequency to 24MHz for the STM32 or 32MHz for the XMEGA and use a repeat 5-10 with both the high power and low power glitches active. You'll need to adjust the baud rate by the same proportion as the clock. This will increase the glitch precision, as the glitch width/offset step is based on the target clock, and may decrease the stability of the target.

Another setup that seems to work with the XMEGA is SJ5 unsoldered, JP6 jumpered, high+low power glitch, 32MHz, and repeat=5.

The ChipWhisperer Husky, with its PLL based glitching system, can typically glitch targets without needing to change the target's frequency.

### Disabling Logging

When glitching (or just running normally in earlier labs), you may have seen various warnings from loggers ChipWhisperer uses. This often has useful information, especially if things don't work right, but for voltage 
glitching especially, it mostly clutters up any print output you have. As such, we'll disable logging for the voltage glitching labs:

```python
cw.set_all_log_levels(cw.logging.CRITICAL)
```

You can reenable logging via

```python
cw.set_all_log_levels(cw.logging.WARNING)
```

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
fw_path = "../../../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER=="SS_VER_2_1":
    target.reset_comms()

In [ ]:
def reboot_flush():
    reset_target(scope)
    target.flush()

if scope._is_husky is False:
    if PLATFORM == "CWLITEXMEGA":
        scope.clock.clkgen_freq = 32E6
        if SS_VER=='SS_VER_2_1':
            target.baud = 230400*32/7.37
        else:
            target.baud = 38400*32/7.37
    elif (PLATFORM == "CWLITEARM") or ("F3" in PLATFORM):
        scope.clock.clkgen_freq = 24E6
        if SS_VER=='SS_VER_2_1':
            target.baud = 230400*24/7.37
        else:
            target.baud = 38400*24/7.37
        time.sleep(0.1)

In [ ]:
reboot_flush()
scope.arm()
target.simpleserial_write("g", bytearray([]))
scope.capture()
val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset", "tries"])
gc.display_stats()

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

Next, we'll setup the glitch. The main differences here compared to clock glitching is the use of `scope.io.glitch_lp/hp`, which activates the transistors used for voltage, and `scope.glitch.output = 'glitch_only'`, which sends the glitch output to the transistors instead of xoring it with the clock like with clock glitching.

In [ ]:
if scope._is_husky:
    scope.glitch.enabled = True
    scope.glitch.clk_src = "pll"
    scope.io.glitch_hp = False
    scope.io.glitch_hp = True
    scope.io.glitch_lp = False
    scope.io.glitch_lp = False
else:
    scope.glitch.clk_src = "clkgen" # set glitch input clock
scope.glitch.output = "glitch_only" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called
if PLATFORM == "CWLITEXMEGA":
    scope.io.glitch_lp = True
    scope.io.glitch_hp = True
elif PLATFORM == "CWLITEARM":
    scope.io.glitch_lp = True
    scope.io.glitch_hp = True
elif PLATFORM == "CW308_STM32F3":
    scope.io.glitch_hp = True
    scope.io.glitch_lp = True

## Part 1: poke the glitcher yourself

The glitch hardware is now set up (previous cell). Before we let the computer search for glitches automatically, **you** will fire some by hand. First run the helper cell below once; it defines the functions the experiments use.

**What can happen when you fire one glitch** (every shot ends in exactly one of these):

| Outcome | What happened | Colour in the plots |
|---|---|---|
| `normal` | The glitch did nothing: the target counted to 2500 as usual. | grey |
| `reset` | The glitch crashed the target. The notebook reboots it and carries on. | red |
| `success` | The glitch changed the answer: the counter is **not** 2500. This is the one we want. | green |

In [ ]:
# ------------------------------------  RUN ONLY (do not edit)  ------------------------------------
# Helper functions for the experiments. Run this cell once; do not edit it.
import time, struct
import numpy as np
import matplotlib.pyplot as plt

cw.set_all_log_levels(cw.logging.CRITICAL)   # keep the output readable
scope.adc.timeout = 0.5                      # a crashed target never finishes: give up quickly
if scope._is_husky:
    scope.adc.lo_gain_errors_disabled = True
    scope.adc.clip_errors_disabled = True

OUTCOME_COLORS = {"normal": "#c9d1d9", "reset": "#d9534f", "success": "#2e9e4f"}


def set_glitch(width, offset, ext_offset, repeat=1):
    """Load one set of glitch settings into the scope."""
    if scope._is_husky:                      # the Husky wants whole numbers (PLL phase-shift steps)
        width, offset = int(round(width)), int(round(offset))
    scope.glitch.width = width
    scope.glitch.offset = offset
    scope.glitch.ext_offset = int(ext_offset)
    scope.glitch.repeat = int(repeat)


def fire_glitch(width, offset, ext_offset, repeat=1):
    """Fire ONE glitch. Returns ("normal" | "reset" | "success", counter value or None)."""
    set_glitch(width, offset, ext_offset, repeat)
    target.flush()
    if scope.adc.state:                      # trigger still high: target is stuck from an earlier crash
        reboot_flush()
    scope.arm()
    target.simpleserial_write("g", bytearray([]))
    timed_out = scope.capture()
    scope.io.vglitch_reset()
    if timed_out:                            # the trigger never came back: the target crashed
        reboot_flush()
        return "reset", None
    val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10, timeout=50)
    if val['valid'] is False:                # no (or a broken) answer: the target crashed
        reboot_flush()
        return "reset", None
    count = struct.unpack("<I", val['payload'])[0]
    return ("normal" if count == 2500 else "success"), count


def fire_many(width, offset, ext_offset, repeat=1, shots=20):
    """Fire `shots` glitches with identical settings. Returns (tally, counters of the successes)."""
    tally = {"normal": 0, "reset": 0, "success": 0}
    counters = []
    cw.set_all_log_levels(cw.logging.CRITICAL)   # the search loop above switched warnings back on: silence them again
    for _ in range(shots):
        outcome, count = fire_glitch(width, offset, ext_offset, repeat)
        tally[outcome] += 1
        if outcome == "success":
            counters.append(count)
    return tally, counters


def scan(points, shots):
    """points = list of (width, offset, ext_offset, repeat). Fire `shots` glitches at each point."""
    tallies, counters = [], []
    for p in points:
        t, c = fire_many(*p, shots=shots)
        tallies.append(t)
        counters += c
        print(".", end="", flush=True)       # one dot per finished point
    print()
    return tallies, counters


def plot_tallies(labels, tallies, xlabel, title):
    """Stacked bars: the share of shots at each x that were normal / reset / success."""
    fig, ax = plt.subplots(figsize=(9, 3.4))
    bottom = np.zeros(len(labels))
    for name in ("success", "reset", "normal"):
        share = np.array([100.0 * t[name] / max(1, sum(t.values())) for t in tallies])
        ax.bar(range(len(labels)), share, bottom=bottom, color=OUTCOME_COLORS[name], label=name, width=0.85)
        bottom += share
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([str(l) for l in labels], rotation=90 if len(labels) > 12 else 0, fontsize=8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("% of shots")
    ax.set_title(title)
    ax.set_ylim(0, 100)
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    plt.tight_layout()
    plt.show()


def print_table(labels, tallies, label_name):
    print("{:>10} {:>8} {:>8} {:>8}".format(label_name, "normal", "reset", "success"))
    for l, t in zip(labels, tallies):
        print("{:>10} {:>8} {:>8} {:>8}".format(str(l), t["normal"], t["reset"], t["success"]))


def crash_edge(labels, tallies, level=0.5):
    """The first label where at least `level` of the shots reset (None if the target never crashed)."""
    for l, t in zip(labels, tallies):
        if t["reset"] >= level * max(1, sum(t.values())):
            return l
    return None


def glitch_on(hp=True, lp=False):
    """Choose which crowbar transistor(s) the glitch drives. (hp, lp) = (False, False) means: no glitch at all."""
    scope.io.glitch_hp = hp
    scope.io.glitch_lp = lp


def recover():
    """Run this if you stopped a cell half way (the square Stop button): reboots the target, puts the glitch hardware back."""
    scope.io.vglitch_reset()
    reboot_flush()
    glitch_on(True, False)
    scope.glitch.repeat = 1
    print("target rebooted, glitch hardware restored (hp on, lp off, repeat 1)")


print("helpers ready: fire_glitch, fire_many, scan, plot_tallies, print_table, crash_edge, glitch_on, recover")

### Experiment 1: fire a glitch by hand

**Goal:** see the three things a glitch can do, and learn what the four knobs mean.

| Knob | What it controls | Husky range |
|---|---|---|
| `WIDTH` | how **long** the glitch is (how long the power is pulled down) | 0 to 2296 |
| `OFFSET` | **where inside a clock cycle** the glitch starts | 0 to 4592 |
| `EXT_OFFSET` | **which clock cycle** after the trigger (0 = the first one) | 0, 1, 2, ... |
| `REPEAT` | how many glitch pulses in a row | 1 to 8192 |

**Do this:** run the next cell as it is. Then change **one number at a time**, run it again, and write down what changed.

| Try | `WIDTH` | `OFFSET` | `EXT_OFFSET` | What do you expect? |
|---|---|---|---|---|
| A (start) | 1770 | 2670 | 1 | ? (this setting is right at the edge: expect big trouble) |
| B: a tiny glitch | 100 | 2670 | 1 | ? |
| C: a huge glitch | 2200 | 2670 | 1 | ? |
| D: the wrong moment | 1770 | 2670 | 30 | ? |
| E: a different offset | 1770 | 1000 | 1 | ? |

**What you should see on our kit:** A gives **successes and resets and hardly any `normal` shots**: between one shot in four and one in three was a success (7 to 10 out of 30), nearly all the rest were resets (the counts change from run to run). B, D and E give only `normal` (the glitch is too weak, or lands at the wrong moment or in the wrong part of the clock cycle). C crashes (`reset`) every time (too strong). Your exact counts will differ.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
WIDTH      = 1770    # glitch length               (Husky: 0 to 2296)
OFFSET     = 2670    # where the glitch starts     (Husky: 0 to 4592)
EXT_OFFSET = 1       # clock cycles after trigger  (0, 1, 2, ...)
REPEAT     = 1       # glitch pulses per shot      (1 to 8192)
SHOTS      = 30      # how many glitches to fire   (more shots = a better estimate, but slower)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

t0 = time.time()
tally, counters = fire_many(WIDTH, OFFSET, EXT_OFFSET, REPEAT, SHOTS)
print("width={} offset={} ext_offset={} repeat={}   |   {} shots in {:.1f} s".format(WIDTH, OFFSET, EXT_OFFSET, REPEAT, SHOTS, time.time() - t0))
for name in ("normal", "reset", "success"):
    print("  {:8s} {:3d}  {}".format(name, tally[name], "#" * tally[name]))
if counters:
    print("counter values after a success:", sorted(set(counters)), "(a normal run always counts to 2500)")
else:
    print("no success this time - try more SHOTS or different numbers")

**My prediction (Experiment 1):** _double-click and write here_

**What I saw:** _..._

### Experiment 2: find the crash edge (scan the width)

**Goal:** find how wide a glitch has to be before it starts to hurt the target, and see why a *good* glitch hides right next to a *crash*.

The next cell fires `SHOTS` glitches at every width from `WIDTH_START` to `WIDTH_STOP` (in steps of `WIDTH_STEP`) and draws one bar per width. Grey = nothing happened, red = the target crashed, green = success.

**Do this**

1. Run the cell as it is (a coarse look at the whole width range). Where do the bars turn red? That is the **crash edge**.
2. **Zoom in** on the edge: set `WIDTH_START = 1700`, `WIDTH_STOP = 1850`, `WIDTH_STEP = 10` and `SHOTS = 20`. Run again. Do you see green now?
3. Change `OFFSET` (for example to 2600, then 2750) and find the edge again. Does it move? Which way?
4. Change `REPEAT` to 2 and find the edge again.

**What you should see:** mostly grey, with **two separate red bands**, not one simple edge you cross and stay past - on our kit, grey from width 0 up to about 1750, red from about 1800-1850, back to **grey again** at 1900-2050, then red again from about 2100 up. Either red band can crash the target; the one near 1800 is the one with a success band hiding just below it (that is where Experiment 1's setting sits). **No green anywhere in this coarse pass** (not even in the second red band, at this offset): the zoomed run around the *first* edge is where green shows up. If your own kit's bands land somewhere else, that is expected - zoom in on whichever red region appears first as you raise the width.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
OFFSET      = 2670    # where the glitch starts (keep fixed while you scan the width)
EXT_OFFSET  = 1
REPEAT      = 1
WIDTH_START = 0       # first width to try   (Husky: 0 to 2296)
WIDTH_STOP  = 2296    # last width to try
WIDTH_STEP  = 100     # distance between widths: smaller = finer, but slower
SHOTS       = 10      # glitches fired at each width
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

widths = list(range(WIDTH_START, WIDTH_STOP + 1, WIDTH_STEP))
print("scanning {} widths x {} shots ...".format(len(widths), SHOTS))
t0 = time.time()
tallies, counters = scan([(w, OFFSET, EXT_OFFSET, REPEAT) for w in widths], SHOTS)
print("done in {:.0f} s".format(time.time() - t0))
plot_tallies(widths, tallies, "WIDTH", "offset={}  ext_offset={}  repeat={}".format(OFFSET, EXT_OFFSET, REPEAT))
print_table(widths, tallies, "width")
edge = crash_edge(widths, tallies)
print("\ncrash edge (first width where at least half the shots crash):", edge)
print("successes:", sum(t["success"] for t in tallies), "  counter values:", sorted(set(counters)))

**My prediction (Experiment 2):** _double-click and write here_

**What I saw (where is the crash edge for OFFSET = 2670? for other offsets?):** _..._

### Experiment 3: find the right moment (scan `EXT_OFFSET`)

**Goal:** a glitch is only useful if it lands on the *right instruction*. `EXT_OFFSET` slides the glitch later in time, one clock cycle per step, after the trigger.

**Do this**

1. Run the cell as it is: it fires at `EXT_START` ... `EXT_STOP`, always with the same width and offset.
2. Which `EXT_OFFSET` values give green? Which give red? Which give only grey?
3. Widen the range (`EXT_STOP = 60`, `SHOTS = 10`) and run again. Does any later cycle give a success?
4. Change `WIDTH` by about 10 either way and repeat. Is the vulnerable cycle still the same?

**What you should see on our kit:** cycle 0 and every cycle from 4 up to about 8 is grey (the glitch lands on something harmless). Cycles 1 to 3 are the dangerous ones: mostly red (crashes), and **only cycle 1 turns green**. Moving the glitch off that cycle makes it fail, so the *timing* matters as much as the *width*. In a wider scan (`EXT_STOP = 60`) we also saw a few red cycles much further out (around 59 and 60), but never another green one.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
WIDTH      = 1770    # use the width / offset that worked in Experiment 1
OFFSET     = 2670
REPEAT     = 1
EXT_START  = 0       # first clock cycle to try
EXT_STOP   = 8       # last clock cycle to try   (try 60 for a wider search)
SHOTS      = 30      # glitches per clock cycle  (the success rate is low, so use plenty)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

exts = list(range(EXT_START, EXT_STOP + 1))
print("scanning {} clock cycles x {} shots ...".format(len(exts), SHOTS))
t0 = time.time()
tallies, counters = scan([(WIDTH, OFFSET, e, REPEAT) for e in exts], SHOTS)
print("done in {:.0f} s".format(time.time() - t0))
plot_tallies(exts, tallies, "EXT_OFFSET (clock cycles after the trigger)", "width={}  offset={}  repeat={}".format(WIDTH, OFFSET, REPEAT))
print_table(exts, tallies, "ext_offset")
good = [e for e, t in zip(exts, tallies) if t["success"]]
print("\nclock cycles with at least one success:", good, "  counter values:", sorted(set(counters)))

**My prediction (Experiment 3):** _double-click and write here_

**What I saw (which cycle is vulnerable? is it the only one?):** _..._

## Part 2: let the computer search

You have now found by hand that good glitches are rare and sit at a precise width, offset and clock cycle. Searching for them one by one is slow, so the lab uses a `GlitchController` that walks through **every combination** in a range and keeps score.

### Experiment 4: run the automatic search with your own ranges

The tips below (read them first) explain how to choose a range. The cell after the tips holds the **numbers you can edit**: the ranges, the step, how many times each setting is tried and how many successes to collect.

**Do this**

1. Run the search with the ranges as they are. Note how long it takes and how many attempts it needed.
2. Make the step bigger (`STEP = 25`), then smaller (`STEP = 1`). What happens to the run time and to the number of successes?
3. Raise `MAX_SUCCESSES` to 5 and run again.
4. Move a range **away** from the good area (for example `WIDTH_RANGE = (1000, 1040)`). What does the plot show?

**What you should see on our kit** (your numbers will differ):

- As it is: the search stops at the first success, printed as `success #1: width = ..., offset = ...`. For us that took a hundred to a few hundred attempts and anywhere from 4 to 15 seconds.
- `STEP = 25`: only a few dozen settings are tried, because the grid skips most values. Whether it happens to land on a good one is **luck**: in one of our runs it found a success in about 1 second, in another it found none and ran through its whole range (about 40 seconds). Run it a few times. A bigger step is faster, but you can step right over the thin band of good settings.
- `STEP = 1`: every value is tried, so nothing is skipped, but there are many more settings. It still stops at the first success (633 attempts, 20 seconds for us); ask for more successes and it takes much longer.
- `MAX_SUCCESSES = 5`: the search keeps going until it has 5 successes, and prints each one as it is found (about 450 attempts and a minute or two for us). `MAX_SUCCESSES = 10` took about 500 attempts and 100 seconds.
- A range far from the good area: no successes at all, and the plot shows only grey dots. Searching in the wrong place wastes time (about 80 seconds for us) and finds nothing.

Some tips for finding good glitches:

1. There's a lot of stuff fighting our glitch this time - unlike the clock line, the Vcc rail isn't supposed to oscillate! As such shorter glitches will have no effect. Often, good widths will be just below when the target starts consistantly crashing, so a good strategy is to find the minimum width where crashes always happen, then backing the width off a bit.
1. The repeat parameter behaves very differently than with voltage glitching - at the boosted clock rate, the Vcc often won't recover before the next glitch. Try different repeat values as well.
1. We've built in a success/reset measurement into the glitch loop. Once you've found some glitch spots, this will help you evaluate which ones are best for your target.

It can take a very long time to do go through the full search space, so you may want to stop after you get a certain number of succeses. By default here, it will be 1, but you may want to change it to 10, 20, or go even higher.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
MAX_SUCCESSES = 1     # stop the search after this many successes (try 5, 10, 20)
NUM_TRIES     = 5     # how many times every (width, offset, ext_offset) is tried

if scope._is_husky:
    # Found by sweeping this exact kit (Husky + CW313 + CW312T-SAM4S, Crowbar SMA connected).
    # Width / offset are Husky phase-shift steps. The successes sit in a thin band at the edge of the crash region.
    WIDTH_RANGE      = (1760, 1800)   # (first, last) width to try
    OFFSET_RANGE     = (2640, 2700)   # (first, last) offset to try
    EXT_OFFSET_RANGE = (0, 3)         # (first, last) clock cycle to try
    STEP             = 5              # distance between neighbouring width / offset values
    REPEAT           = 1              # glitch pulses per shot
elif PLATFORM == "CWLITEXMEGA":
    WIDTH_RANGE, OFFSET_RANGE, EXT_OFFSET_RANGE, STEP, REPEAT = (43.5, 47.8), (-48, -10), (7, 10), 0.4, 11
elif PLATFORM == "CWLITEARM":
    WIDTH_RANGE, OFFSET_RANGE, EXT_OFFSET_RANGE, STEP, REPEAT = (34, 36), (-40, 10), (4, 30), 0.4, 7
elif PLATFORM == "CW308_STM32F3":
    WIDTH_RANGE, OFFSET_RANGE, EXT_OFFSET_RANGE, STEP, REPEAT = (47.6, 49.6), (-19, -21.5), (4, 30), 0.4, 5
else:
    raise NotImplementedError("No glitch ranges for PLATFORM = " + PLATFORM + ": choose your own above")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

In [ ]:
import time
import struct

#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)

# The glitch ranges (width, offset, ext_offset, step, repeat) come from the EDIT ME cell above.
gc.set_range("tries", 0, NUM_TRIES)
gc.set_range("width", *WIDTH_RANGE)
gc.set_range("offset", *OFFSET_RANGE)
gc.set_range("ext_offset", *EXT_OFFSET_RANGE)
gc.set_global_step([STEP])
gc.set_step("tries", 1)
gc.set_step("ext_offset", 1)
scope.glitch.repeat = REPEAT
if scope._is_husky:
    scope.adc.lo_gain_errors_disabled = True
    scope.adc.clip_errors_disabled = True

scope.adc.timeout = 0.5

reboot_flush()
loff = scope.glitch.offset
lwid = scope.glitch.width
total_successes = 0
attempts = 0
search_start = time.time()
successes = 0
resets = 0
for glitch_setting in gc.glitch_values():
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    scope.glitch.ext_offset = glitch_setting[2]
    if glitch_setting[3] == 0:
        successes = 0
        resets = 0
    if total_successes >= MAX_SUCCESSES:
        break
    attempts += 1
    target.flush()
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        #print("Trigger still high!")
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
        resets += 1

    scope.arm()

    #Do glitch loop
    target.simpleserial_write("g", bytearray([]))

    ret = scope.capture()

    scope.io.vglitch_reset()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")
        resets += 1

        #Device is slow to boot?
        reboot_flush()

    else:
        val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10, timeout=50)#For loop check
        if val['valid'] is False:
            gc.add("reset")
            reboot_flush()
            resets += 1
            #print(val)
        else:
            gcnt = struct.unpack("<I", val['payload'])[0]

            if gcnt != 2500: #for loop check
                gc.add("success")
                successes += 1
                total_successes += 1
                print("success #{}: width = {}, offset = {}, ext_offset = {}   (counter = {})".format(
                    total_successes, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset, gcnt))
            else:
                gc.add("normal")
print("Done glitching: {} successes, {} attempts, {:.0f} s".format(total_successes, attempts, time.time() - search_start))
#enable logging
cw.set_all_log_levels(cw.logging.WARNING)

With that done, let's print our results. You'll likely want to ignore the "tries" and "ext_offset" parameters:

In [ ]:
results = gc.calc(ignore_params=["tries", "ext_offset"], sort="success_rate")
results

**How to read the table above.** Each entry is `((width, offset), {...})`: one width/offset pair, totalled over all `ext_offset` and `tries` values. `total` is how many glitches were fired, and `success`, `reset` and `normal` count the outcomes (the `_rate` values are the same as fractions of `total`). The list is sorted so the **best (width, offset) pair, the one with the highest `success_rate`, comes first**.

## Part 3: map it, strengthen it, measure it

### Experiment 5: draw a crash map (width x offset)

**Goal:** see the whole picture at once. The cell fires glitches over a grid of widths and offsets and plots one mark per shot: grey dot = normal, red cross = reset, green plus = success.

**Do this**

1. Run the cell as it is (about a minute). Look at the **shape** of the red region: where does it start? Does the edge run straight up, or does it lean?
2. Is there any offset where a *very* wide glitch is still harmless? What does that tell you about where, inside a clock cycle, the glitch has to land?
3. Zoom in on the edge: choose a small window with a small step (for example `WIDTH_START, WIDTH_STOP, WIDTH_STEP = 1700, 1850, 10` and `OFFSET_START, OFFSET_STOP, OFFSET_STEP = 2600, 2750, 10`, `SHOTS = 3`; this takes about 4 minutes). Where are the green marks?
4. Pick three green marks and add up **WIDTH + OFFSET** for each. What do you notice?

**What you should see on our kit:** the first run is grey almost everywhere (glitch too weak, or the wrong part of the clock cycle) with a red patch for wide glitches at offsets between about 2700 and 3000. In the zoomed run the red patch has a **slanted lower-left edge**, and the green marks lie in a thin line **right along that edge**, just outside the red. In our run the sum WIDTH + OFFSET was about 4430 to 4480 for every green mark: the glitch has to be *long enough* and *early enough in the cycle* in just the right combination.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
EXT_OFFSET   = 1
REPEAT       = 1
WIDTH_START, WIDTH_STOP, WIDTH_STEP      = 1500, 2100, 50      # Husky: 0 to 2296
OFFSET_START, OFFSET_STOP, OFFSET_STEP   = 2000, 3200, 100     # Husky: 0 to 4592
SHOTS        = 2       # glitches per grid point (more = clearer, slower)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

widths = list(range(WIDTH_START, WIDTH_STOP + 1, WIDTH_STEP))
offsets = list(range(OFFSET_START, OFFSET_STOP + 1, OFFSET_STEP))
print("map: {} widths x {} offsets x {} shots = {} glitches".format(len(widths), len(offsets), SHOTS, len(widths) * len(offsets) * SHOTS))
cw.set_all_log_levels(cw.logging.CRITICAL)   # keep the output readable
t0 = time.time()
shots_log = []
for w in widths:
    for o in offsets:
        for _ in range(SHOTS):
            outcome, count = fire_glitch(w, o, EXT_OFFSET, REPEAT)
            shots_log.append((w, o, outcome))
    print(".", end="", flush=True)
print("\ndone in {:.0f} s".format(time.time() - t0))

fig, ax = plt.subplots(figsize=(7, 5))
for outcome, marker, size in (("normal", ".", 25), ("reset", "x", 40), ("success", "P", 70)):
    pts = [(w, o) for (w, o, r) in shots_log if r == outcome]
    if pts:
        ax.scatter([p[0] for p in pts], [p[1] for p in pts], marker=marker, s=size, c=OUTCOME_COLORS[outcome] if outcome != "normal" else "#9aa5b1",
                   label="{} ({})".format(outcome, len(pts)))
ax.set_xlabel("WIDTH")
ax.set_ylabel("OFFSET")
ax.set_title("crash map: ext_offset={}  repeat={}".format(EXT_OFFSET, REPEAT))
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout()
plt.show()
print("successes:", [(w, o) for (w, o, r) in shots_log if r == "success"])

**My prediction (Experiment 5):** _double-click and write here_

**What I saw (what is WIDTH + OFFSET at the green marks?):** _..._

### Experiment 6: turn the glitch up and down (`REPEAT` and the HP / LP transistors)

**Goal:** the glitch is made by switching a transistor that shorts the target's power rail to ground. You can try to make the glitch *stronger* by repeating the pulse (`REPEAT`) or by using more transistors. A stronger glitch should crash the target at a **smaller width**: the crash edge moves left. Does it?

| Setting | What it means |
|---|---|
| `hp` | the **high-power** crowbar transistor |
| `lp` | the **low-power** crowbar transistor |
| both off | no crowbar at all: the glitch has nowhere to go (a **control experiment**: nothing should happen) |

**Do this**

1. Run the cell as it is. For every setting in `SETTINGS` it scans the width and reports the crash edge (the first width where half of the shots crash).
2. Compare the edges. Does `REPEAT` 2 or 4 move the edge? Does HP + LP? What does the control (both off) do?
3. Add a line of your own to `SETTINGS`, for example `("hp, repeat 8", 8, True, False)`.

**What you should see on our kit** (write down your own numbers; they can differ by a step or two):

- `hp only` with `REPEAT` 1, 2 and 4: the crash edge stays at **about the same width** (1800 in our run). Repeating the pulse did **not** make it stronger here.
- `hp + lp`: the edge moves to a **smaller width** (about 1750): two transistors really are stronger.
- `lp only`: **never crashes** in this range. The low-power transistor alone is too weak.
- both off (the control): **never crashes**. No transistor, no glitch.

A surprise like "repeat did nothing" is a result, not a mistake: write it down and try to explain it.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
OFFSET      = 2670
EXT_OFFSET  = 1
WIDTH_START = 1500      # scan this range of widths for every setting ...
WIDTH_STOP  = 2100
WIDTH_STEP  = 50
SHOTS       = 3         # ... firing this many glitches at each width

# (name, repeat, hp, lp)  - add or remove lines as you like
SETTINGS = [
    ("hp only, repeat 1",  1, True,  False),
    ("hp only, repeat 2",  2, True,  False),
    ("hp only, repeat 4",  4, True,  False),
    ("lp only, repeat 1",  1, False, True),
    ("hp + lp, repeat 1",  1, True,  True),
    ("both OFF (control)", 1, False, False),
]
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

widths = list(range(WIDTH_START, WIDTH_STOP + 1, WIDTH_STEP))
results = {}
t0 = time.time()
try:
    for name, repeat, hp, lp in SETTINGS:
        glitch_on(hp, lp)
        print("{:20s}".format(name), end=" ", flush=True)
        tallies, counters = scan([(w, OFFSET, EXT_OFFSET, repeat) for w in widths], SHOTS)
        results[name] = (tallies, counters)
finally:
    glitch_on(True, False)           # always put the normal glitch setup back
    scope.glitch.repeat = 1
print("done in {:.0f} s\n".format(time.time() - t0))

print("{:22s} {:>12} {:>10}".format("setting", "crash edge", "successes"))
fig, ax = plt.subplots(figsize=(9, 3.6))
for name, (tallies, counters) in results.items():
    edge = crash_edge(widths, tallies)
    print("{:22s} {:>12} {:>10}".format(name, str(edge), sum(t["success"] for t in tallies)))
    ax.plot(widths, [100.0 * t["reset"] / max(1, sum(t.values())) for t in tallies], marker="o", ms=3, label=name)
ax.set_xlabel("WIDTH")
ax.set_ylabel("% of shots that crashed")
ax.set_title("how often the target crashes, per setting (offset={}, ext_offset={})".format(OFFSET, EXT_OFFSET))
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout()
plt.show()

**My prediction (Experiment 6):** _double-click and write here_

**What I saw (which setting crashes the target first? what did the control do?):** _..._

### Experiment 7: how reliable is your best glitch?

**Goal:** a glitch that worked once may not work again. A real attack needs to know **how often** it works, because that decides how many tries (and how many crashes) it costs. This cell fires many shots at a few neighbouring settings and compares them.

**Do this**

1. Run the cell as it is. Which candidate has the best success rate?
2. Replace the candidates with your **own** best settings from Experiments 2 to 5.
3. Raise `SHOTS` to 200 for the winner. Does its rate stay the same? (Run the cell twice: how much does the rate change by luck alone?)

**What you should see on our kit:** success rates between 0% and about 40%. Neighbouring settings differ a lot: moving the width or offset by only 10 steps can take a setting from "works a third of the time" to "never works, always crashes". The same setting also gives a slightly different rate each time you measure it, so a single run of 20 shots tells you very little.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
SHOTS = 50

# (width, offset, ext_offset, repeat) - put your own candidates here
CANDIDATES = [
    (1760, 2680, 1, 1),
    (1770, 2680, 1, 1),
    (1780, 2680, 1, 1),
    (1770, 2670, 1, 1),
    (1770, 2690, 1, 1),
]
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

t0 = time.time()
tallies, counters = scan(CANDIDATES, SHOTS)
print("done in {:.0f} s\n".format(time.time() - t0))
print("{:>28} {:>9} {:>8} {:>8} {:>18}".format("width, offset, ext, repeat", "success", "reset", "normal", "tries per success"))
for c, t in zip(CANDIDATES, tallies):
    rate = t["success"] / SHOTS
    print("{:>28} {:>8.0f}% {:>8} {:>8} {:>18}".format(str(c), 100 * rate, t["reset"], t["normal"], "{:.0f}".format(1 / rate) if rate else "never"))
print("\ncounter values seen on success:", sorted(set(counters)))

**My prediction (Experiment 7):** _double-click and write here_

**What I saw (best candidate, its success rate, tries per success):** _..._

## Challenges (optional)

Use the helper functions above (`fire_glitch`, `fire_many`, `scan`, `plot_tallies`, ...) and the **EDIT ME** cells.

1. **The cheapest success.** Set `MAX_SUCCESSES = 10` in the search and find ranges and a `STEP` that collect 10 successes in the **fewest attempts** and the **least time**. Write down your settings and the run time.
2. **Why always the same number?** Look at the counter values you saw after a success. Is it always the same? The firmware counts to 2500 (50 x 50). What does it tell you about *what* the glitch changed if the answer is the same every time?
3. **A different target cycle.** Experiment 3 found one vulnerable cycle. Use a wider scan and different widths and offsets: can you find a *second* cycle that gives a success, or a cycle that only ever crashes the target?
4. **Quiet glitching.** Every `reset` is a crashed target, which a real defender could notice. Find a setting with a success rate of at least 5% and **fewer than 1 reset per success**.

## If something goes wrong

| What you see | Most likely cause | What to do |
|---|---|---|
| Every shot is `normal`, never a `reset` or `success`, even with a huge width | The Husky **Crowbar** SMA cable is not connected to the CW313's second SMA socket | Fit the cable (see the hardware checklist at the top), run the setup cells again |
| **Every** shot is `reset` | The width is too big for this offset, or the target is stuck from an earlier crash | Lower `WIDTH` (try 1700), run `recover()` in a new cell, try again |
| Plenty of `reset`, but no `success` | The numbers are slightly off: successes live in a band only a few steps wide, at the edge of the crash region | Go back to Experiment 2 and zoom in with a step of 5 or 10; use more `SHOTS` |
| A `success` shows up in one run and not in the next | Normal: the success rate is a probability (see Experiment 7) | Fire more shots and compare *rates*, not single results |
| You pressed **Stop** (square button) or a cell was interrupted | The target may be halfway through a crash and the glitch hardware in an odd state | Run `recover()` in a new cell |
| `OSError: Could not find ChipWhisperer` or `Unable to communicate with device` | The Husky is not plugged in, or another notebook still holds it | Plug it in; close every other notebook that uses it (**File > Shut Down Notebook**), then restart this kernel (**Kernel > Restart**) |
| `USBErrorPipe`, or `Unknown hwInfoVer` / `ValueError: Unknown ChipWhisperer` | The Husky's USB link or FPGA has stopped answering | **Unplug the Husky USB cable and plug it back in** (straight into the computer, not through a hub), then restart the kernel and run the setup cells again |
| The search or a scan takes a very long time | Every `reset` costs about half a second (the target has to reboot) | Use fewer widths (bigger step), fewer `SHOTS`, or a narrower range |
| Strange numbers or an error right after you edit | A typo, or a value outside the Husky's range (`WIDTH` 0 to 2296, `OFFSET` 0 to 4592) | Put the original numbers back (they are in the markdown above each cell) and change one thing at a time |

Stay inside the ranges quoted in the experiments, and **do not unplug any cable while a cell is running**. A crashed target is normal here: the notebook reboots it for you.

## Finish: switch the glitch hardware off

The last cell switches the glitch transistors off and disconnects the Husky and the target, so the board is safe to leave. **Always run it before closing the notebook.**

In [ ]:
if scope._is_husky:
    # switch the glitch hardware off before disconnecting
    scope.io.glitch_hp = False
    scope.io.glitch_lp = False
    scope.glitch.enabled = False
scope.dis()
target.dis()